# Titian — record-level data provenance

Run a query with capture on, then trace an output record back to the exact input
records that produced it. Provenance answers *which records produced this*.

In [ ]:
import os, sys, glob

ROOT = os.environ.get("BIGASTERISK_HOME") or os.path.abspath("..")

# Jars: a source checkout has them under modules/*/target, the Docker image under jars/.
JARS = sorted(glob.glob(f"{ROOT}/modules/*/target/scala-2.13/bigasterisk-*.jar")) \
    or sorted(glob.glob(f"{ROOT}/jars/bigasterisk-*.jar"))
if not JARS:
    raise SystemExit("No BigAsterisk jars found. Run: bin/sbt package")

FASTUTIL_JAR = os.environ.get("FASTUTIL_JAR") or next(iter(sorted(
    glob.glob(f"{ROOT}/jars/fastutil*.jar")
    + glob.glob(os.path.expanduser("~/Library/Caches/Coursier/**/fastutil-8.5.15.jar"), recursive=True)
    + glob.glob(os.path.expanduser("~/.cache/coursier/**/fastutil-8.5.15.jar"), recursive=True)
)), None)
if not FASTUTIL_JAR:
    raise SystemExit("fastutil jar not found. Run: bin/sbt package")

SPARK_JARS = ",".join(JARS + [FASTUTIL_JAR])
DATA = f"{ROOT}/examples/data"
sys.path.insert(0, f"{ROOT}/python")

## The data

Twelve orders across three customers. One of them, `o8`, is an outlier at
`99999` — every notebook here uses it as the thing to find.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import bigasterisk

spark = (bigasterisk.configure(SparkSession.builder)
    .master("local[2]")
    .appName("titian-notebook")
    .config("spark.jars", SPARK_JARS)
    .config("spark.sql.adaptive.skewJoin.enabled", "false")
    .config("spark.ui.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

orders = spark.read.schema("oid STRING, cid STRING, amount INT").csv(f"{DATA}/orders.txt")
customers = spark.read.schema("cid STRING, name STRING").csv(f"{DATA}/customers.txt")
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

orders.show()

## Capture and trace

The totals are grouped, so each output row has several records behind it.

In [ ]:
lineage = bigasterisk.lineage(spark)
lineage.enable_capture()

df = spark.sql("SELECT cid, SUM(amount) AS total FROM orders GROUP BY cid")
rows = lineage.collect_with_lineage(df)
for row, rid in rows:
    print(row, rid)

## Trace one output back to its inputs

In [ ]:
big = max(rows, key=lambda r: r[0]["total"])
print("tracing:", big[0])

witnesses = lineage.trace(df, [big[1]]).to_scan().show(full=True)
for w in witnesses:
    print(" ", w)

## Check

Every witness belongs to the traced group, and they sum to its total.

In [ ]:
assert all(w["cid"] == big[0]["cid"] for w in witnesses), witnesses
assert sum(w["amount"] for w in witnesses) == big[0]["total"]
lineage.release_lineage(df)
print("OK")